# 🧠 LLM from Scratch — 01: Data & Tokenization

First steps toward building a language model from scratch:

1. ⚙️ **Setup**: install packages and import everything
2. 📚 **Dataset**: explore *TinyStories*
3. 🤖 **Pretrained baseline**: generate a story with `TinyStories-33M`
4. 📖 **Vocabulary**: what a real tokenizer vocab looks like
5. ✂️ **Tokenization**: whitespace → regex → NLTK → spaCy → Gensim → 🤗 BPE
6. 📊 **Comparison & takeaways**

---
## ⚙️ 1. Setup

### 📦 Install packages

In [1]:
%pip install -q datasets gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 56.3 MB/s eta 0:00:00:00:0100:01


### 📥 Imports

In [44]:
import json
import random
import re
from pprint import pprint

import pandas as pd
import torch
import torch.nn as nn

# 🤗 Hugging Face
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from tokenizers import Tokenizer, normalizers, pre_tokenizers, models, processors, decoders, trainers
from transformers import AutoModelForCausalLM, AutoTokenizer

# 🔤 Classic NLP tokenizers
import nltk
import spacy
from gensim.utils import simple_preprocess
from nltk.tokenize import RegexpTokenizer, word_tokenize

---
## 📚 2. Dataset: TinyStories

[`roneneldan/TinyStories`](https://huggingface.co/datasets/roneneldan/TinyStories) is a set of
short stories generated by GPT-3.5/GPT-4 using only words a 3–4 year old would understand.
Its small vocabulary and simple grammar make it a good dataset for training small LMs from scratch.

In [3]:
dataset = load_dataset("roneneldan/TinyStories")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…): reconstructing file:   0%|          |  0.00B /  249MB            

data/train-00000-of-00004-2d5a1467fff108(…): downloading bytes:           |  0.00B            

data/train-00001-of-00004-5852b56a2bd28f(…): reconstructing file:   0%|          |  0.00B /  248MB            

...

data/train-00003-of-00004-d243063613e5a0(…): downloading bytes:           |  0.00B            

data/validation-00000-of-00001-869c898b5(…): reconstructing file:   0%|          |  0.00B / 9.99MB            

data/validation-00000-of-00001-869c898b5(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

In [4]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 2119719
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 21990
    })
})

### 🔍 A sample from the validation split

In [5]:
pprint(dataset['validation']['text'][0])

('Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright '
 'and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every '
 'day."\n'
 '\n'
 'After playing with the car, Kitty and Spot felt thirsty. They found a small '
 'pond with clear water. They drank the water and felt very happy. They played '
 'together all day and became best friends.')


### 🎲 Random sample helper

In [30]:
def get_sample(ds, split="train"):
    """Return a random story from the given split."""
    idx = random.randrange(len(ds[split]))
    return ds[split][idx]["text"]

In [5]:
get_sample(dataset)

'Once upon a time, in a big garden, there was a thin ant named Andy. Andy liked to play in the sun. The sun would shine and make everything warm and bright.\n\nOne day, Andy met a big bug named Betty. Betty was not nice. She said, "Andy, you are too small and thin to play with me!" Andy felt sad and went to sit under a leaf.\n\nThen, a kind ladybug named Lucy came to Andy. She said, "Don\'t be sad, Andy. You may be thin, but you are strong and fast. Let\'s show Betty how good you are!" Andy and Lucy went back to Betty.\n\nAndy said, "Betty, I am thin, but I can do a lot of things. Watch me!" Andy ran fast, climbed high, and carried a big crumb. Betty was surprised. She said, "Wow, Andy, I didn\'t know you could do all that. I am sorry. Can we be friends?" Andy, Lucy, and Betty became friends and played together in the shining sun.'

---
## 🤖 3. Pretrained Baseline: TinyStories-33M

Before building our own model, let's see what a small model trained on TinyStories can do.
`TinyStories-33M` is a GPT-Neo model with ~33M parameters, and it uses the GPT-Neo tokenizer.

In [10]:
gpt_tokenizer = AutoTokenizer.from_pretrained("EleutherAI/gpt-neo-125M")
model = AutoModelForCausalLM.from_pretrained("roneneldan/TinyStories-33M", device_map="auto")

config.json:   0%|          | 0.00/1.01k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/56 [00:00<?, ?it/s]

[transformers] GPTNeoForCausalLM LOAD REPORT from: roneneldan/TinyStories-33M
Key                                                   | Status     |  | 
------------------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3}.attn.attention.bias        | UNEXPECTED |  | 
transformer.h.{0, 1, 2, 3}.attn.attention.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors: reconstructing file:   0%|          |  0.00B /  291MB            

model.safetensors: downloading bytes:           |  0.00B            

### ✍️ Generate a story

In [11]:
prompt = "One day, a little girl named"

input_ids = gpt_tokenizer.encode(prompt, return_tensors="pt").to(model.device)
output = model.generate(input_ids, max_length=500, num_beams=1)

pprint(gpt_tokenizer.decode(output[0], skip_special_tokens=True))

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


('One day, a little girl named Lily went to the park with her mom. They saw a '
 'big tree with a swing hanging from it. Lily wanted to play on the swing, but '
 'it was too high for her to reach.\n'
 '\n'
 'Lily\'s mom said, "Don\'t worry, I will help you." She picked her up and put '
 'her on the swing. Lily was so happy! She started to swing back and forth, '
 'higher and higher.\n'
 '\n'
 'As Lily was swinging, she saw a little boy named Tim. Tim was sad because he '
 'lost his toy. Lily stopped swinging and went to help Tim. She said, "Don\'t '
 'be sad, Tim. We will find your toy." They looked and looked, and finally, '
 'they found the toy under a bush. Tim was so happy, and he said, "Thank you, '
 'Lily!"\n'
 '\n'
 'Lily and Tim became good friends. They played together at the park every '
 'day. They always helped each other and had lots of fun. And they always '
 'remembered to be careful when they played on the swing.\n')


---
## 📖 4. A Real Tokenizer Vocabulary

Download `vocab.json` from `TinyStories-8M`. It maps every **token → id**.
The first ids are single bytes (`!`, `"`, `#`, …, `A`–`Z`, `a`–`z`, …). Later ids are merged sub-words.

In [6]:
vocab_path = hf_hub_download(repo_id="roneneldan/TinyStories-8M", filename="vocab.json")

with open(vocab_path) as f:
    tiny_vocab = json.load(f)  # {token: id}

tokens = list(tiny_vocab.items())
print(f"vocab size: {len(tiny_vocab):,}")
print("first ids :", tokens[:20])
print("later ids :", tokens[5_000:5_020])

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

vocab size: 50,257
first ids : [('!', 0), ('"', 1), ('#', 2), ('$', 3), ('%', 4), ('&', 5), ("'", 6), ('(', 7), (')', 8), ('*', 9), ('+', 10), (',', 11), ('-', 12), ('.', 13), ('/', 14), ('0', 15), ('1', 16), ('2', 17), ('3', 18), ('4', 19)]
later ids : [('Ġentirely', 5000), ('Ġpurchase', 5001), ('Ġelement', 5002), ('Ġcash', 5003), ('Ġdetermine', 5004), ('De', 5005), ('Ġcars', 5006), ('ĠWall', 5007), ('âĸ', 5008), ('Ġviews', 5009), ('Ġdrugs', 5010), ('Ġdepartment', 5011), ('ĠStep', 5012), ('uit', 5013), ('Ġ39', 5014), ('asure', 5015), ('ĠClass', 5016), ('Ġcovered', 5017), ('ĠBank', 5018), ('Ġmere', 5019)]


---
## ✂️ 5. Tokenization

**Tokenization** splits raw text into units (*tokens*) that a model can map to ids.
We compare several approaches on the same tricky sentence, which has an abbreviation (`Mr.`),
a domain (`cheapsite.com`), a price, and a year:

In [7]:
sentence = "Mr. Smith bought cheapsite.com for $1.5 million in 2023."

### ⬜ 5.1 Whitespace tokenization

The simplest approach: split on spaces. Punctuation stays glued to words.

In [ ]:
text = 'Lorem Ipsum has been the industry standard dummy text ever since 1966'

text.split()

['Lorem',
 'Ipsum',
 'has',
 'been',
 'the',
 'industry',
 'standard',
 'dummy',
 'text',
 'ever',
 'since',
 '1966']

### 🧩 5.2 Regex tokenization

Put spaces around punctuation, then split on whitespace.

In [ ]:
PUNCT = r'([\.,!?;:\(\)"])'

def regex_tokenize(text):
    return re.sub(PUNCT, r' \1 ', text).split()


text = "Hello world! This is a simple example of whitespace tokenization."

print(text.split())
print(regex_tokenize(text))

['Hello', 'world!', 'This', 'is', 'a', 'simple', 'example', 'of', 'whitespace', 'tokenization.']
['Hello', 'world', '!', 'This', 'is', 'a', 'simple', 'example', 'of', 'whitespace', 'tokenization', '.']


In [ ]:
print(sentence.split())
print(regex_tokenize(sentence))

['Mr.', 'Smith', 'bought', 'cheapsite.com', 'for', '$1.5', 'million', 'in', '2023.']
['Mr', '.', 'Smith', 'bought', 'cheapsite', '.', 'com', 'for', '$1', '.', '5', 'million', 'in', '2023', '.']


> ⚠️ Regex splits on **every** `.`, so it breaks `Mr.`, `cheapsite.com` and the number `1.5`.

### 📗 5.3 NLTK

- `RegexpTokenizer(r'\w+')`: keeps only word characters and drops all punctuation.
- `word_tokenize`: uses the Punkt model with Treebank rules, so it handles abbreviations, domains and decimals.

In [ ]:
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
regexp_tokenizer = RegexpTokenizer(r'\w+')

print(regexp_tokenizer.tokenize(sentence))
print(word_tokenize(sentence))

['Mr', 'Smith', 'bought', 'cheapsite', 'com', 'for', '1', '5', 'million', 'in', '2023']
['Mr.', 'Smith', 'bought', 'cheapsite.com', 'for', '$', '1.5', 'million', 'in', '2023', '.']


### 🚀 5.4 spaCy

A rule-based tokenizer with language-specific exceptions. Each token is a rich `Token`
object with attributes such as `.pos_` and `.lemma_`.

In [ ]:
nlp = spacy.load('en_core_web_sm')

doc = nlp(sentence)
list(doc)

[Mr., Smith, bought, cheapsite.com, for, $, 1.5, million, in, 2023, .]

### 🧪 5.5 Gensim

`simple_preprocess` lowercases the text, strips accents (`deacc=True`), keeps only alphabetic
tokens, and drops very short ones. Numbers and symbols disappear.

In [ ]:
simple_preprocess(sentence, deacc=True)

['mr', 'smith', 'bought', 'cheapsite', 'com', 'for', 'million', 'in']

### 🤗 5.6 Hugging Face `tokenizers`: sub-word BPE (DeepSeek-V3)

Modern LLMs use **byte-level BPE**:

- Rare words are split into sub-word pieces: `cheapsite` → `Ġche` + `aps` + `ite`.
- `Ġ` marks a **leading space**, so no information is lost.
- Numbers are split into small digit chunks.

In [15]:
hf_tokenizer = Tokenizer.from_pretrained("deepseek-ai/DeepSeek-V3-0324")

encoding = hf_tokenizer.encode(sentence)
pd.DataFrame({"token": encoding.tokens, "id": encoding.ids})

tokenizer.json:   0%|          | 0.00/7.85M [00:00<?, ?B/s]

,token,id
0,Mr,16735
1,.,16
2,ĠSmith,10201
3,Ġbought,16180
4,Ġche,1639
5,aps,3471
6,ite,845
7,.com,2193
8,Ġfor,362
9,Ġ$,957


#### 🔁 Decoding is lossless

In [16]:
hf_tokenizer.decode(encoding.ids)

'Mr. Smith bought cheapsite.com for $1.5 million in 2023.'

#### 📖 Exploring the vocabulary

Strings like `ãģĹãģŁ` are UTF-8 **bytes** (here, of CJK text) shown through the
byte-to-unicode mapping used by byte-level BPE.

In [17]:
vocab = hf_tokenizer.get_vocab()  # {token: id}
len(vocab)

128815

In [18]:
sorted_vocab = sorted(vocab.items(), key=lambda item: item[1])  # sort by token id

print("first ids  :", sorted_vocab[:30])            # special tokens + single bytes
print("middle ids :", sorted_vocab[10_000:10_015])  # merged sub-words
print("last ids   :", sorted_vocab[-10:])

first ids  : [('<｜begin▁of▁sentence｜>', 0), ('<｜end▁of▁sentence｜>', 1), ('<｜▁pad▁｜>', 2), ('!', 3), ('"', 4), ('#', 5), ('$', 6), ('%', 7), ('&', 8), ("'", 9), ('(', 10), (')', 11), ('*', 12), ('+', 13), (',', 14), ('-', 15), ('.', 16), ('/', 17), ('0', 18), ('1', 19), ('2', 20), ('3', 21), ('4', 22), ('5', 23), ('6', 24), ('7', 25), ('8', 26), ('9', 27), (':', 28), (';', 29)]
middle ids : [('åĪ·', 10000), ('ÑĢÐ°Ð½', 10001), ('æĪĲç»©', 10002), ('Ġuntuk', 10003), ('Ġhat', 10004), ('Ġking', 10005), ('aming', 10006), ('ramework', 10007), ('ĠOper', 10008), ('Reg', 10009), ('éĹª', 10010), ('ĠQue', 10011), ('Ġsed', 10012), ('enced', 10013), ('uster', 10014)]
last ids   : [('<|EOT|>', 128805), ('<｜tool▁calls▁begin｜>', 128806), ('<｜tool▁calls▁end｜>', 128807), ('<｜tool▁call▁begin｜>', 128808), ('<｜tool▁call▁end｜>', 128809), ('<｜tool▁outputs▁begin｜>', 128810), ('<｜tool▁outputs▁end｜>', 128811), ('<｜tool▁output▁begin｜>', 128812), ('<｜tool▁output▁end｜>', 128813), ('<｜tool▁sep｜>', 128814)]


---
## 📊 6. Comparison

Results on `sentence`:

| Method | # | Tokens |
|---|:-:|---|
| `str.split()` | 9 | `Mr.` `Smith` `bought` `cheapsite.com` `for` `$1.5` `million` `in` `2023.` |
| Regex | 15 | `Mr` `.` `Smith` `bought` `cheapsite` `.` `com` `for` `$1` `.` `5` `million` `in` `2023` `.` |
| NLTK `RegexpTokenizer` | 11 | `Mr` `Smith` `bought` `cheapsite` `com` `for` `1` `5` `million` `in` `2023` |
| NLTK `word_tokenize` | 11 | `Mr.` `Smith` `bought` `cheapsite.com` `for` `$` `1.5` `million` `in` `2023` `.` |
| spaCy | 11 | `Mr.` `Smith` `bought` `cheapsite.com` `for` `$` `1.5` `million` `in` `2023` `.` |
| Gensim | 8 | `mr` `smith` `bought` `cheapsite` `com` `for` `million` `in` |
| 🤗 DeepSeek-V3 BPE | 19 | `Mr` `.` `ĠSmith` `Ġbought` `Ġche` `aps` `ite` `.com` `Ġfor` `Ġ$` `1` `.` `5` `Ġmillion` `Ġin` `Ġ` `202` `3` `.` |

### ✅ Takeaways

- 🚀 **spaCy**: production-grade NLP pipelines where accuracy matters.
- 📗 **NLTK**: learning and lightweight preprocessing.
- 🧪 **Gensim**: topic modeling and document similarity, where lossy preprocessing is fine.
- 🤗 **Sub-word BPE**: what LLMs actually use. It has a fixed vocab size, no out-of-vocabulary words, and it is fully reversible.

#### traini a bpe tokenizer

In [9]:
ds = [
    "low lower new newest widest",
    "low lower new new wider",
    "lowest wider wider newest",
]

In [25]:
tokenizer = Tokenizer(models.BPE(unk_token="[UNK]"))
tokenizer

trainer = trainers.BpeTrainer(
    vocab_size = 40,
    special_tokens=["[UNK]"],
    min_frequency=2
)

tokenizer.train_from_iterator(ds, trainer)

tokenizer.save("bpe_tokenizer.json")

In [26]:
tokenizer.encode('bellow lowest new').tokens

['[UNK]', 'e', 'l', 'low low', 'est', ' new']

In [27]:
tokenizer.get_vocab()

{' low': 24,
 'er': 13,
 ' wid': 21,
 'low': 17,
 '[UNK]': 0,
 ' w': 18,
 'id': 20,
 'd': 2,
 ' n': 12,
 ' ': 1,
 'n': 6,
 'lo': 15,
 's': 9,
 ' new': 16,
 'i': 4,
 'r': 8,
 'low lower new new': 28,
 'er new new': 27,
 'w': 11,
 'es': 19,
 't': 10,
 'low low': 25,
 'e': 3,
 ' wider': 26,
 'l': 5,
 'er new': 23,
 'ew': 14,
 'o': 7,
 'est': 22}

In [16]:
vocab = tokenizer.get_vocab()
sorted_vocab = dict(sorted(vocab.items(), key=lambda item: item[1]))
sorted_vocab

{'[UNK]': 0,
 ' ': 1,
 'd': 2,
 'e': 3,
 'i': 4,
 'l': 5,
 'n': 6,
 'o': 7,
 'r': 8,
 's': 9,
 't': 10,
 'w': 11,
 ' n': 12,
 'er': 13,
 'ew': 14,
 'lo': 15,
 ' new': 16,
 'low': 17,
 ' w': 18,
 'es': 19,
 'id': 20,
 ' wid': 21,
 'est': 22,
 'er new': 23,
 ' low': 24,
 'low low': 25,
 ' wider': 26,
 'er new new': 27,
 'low lower new new': 28}

In [32]:
dataset['validation']['text'][0]

'Spot. Spot saw the shiny car and said, "Wow, Kitty, your car is so bright and clean!" Kitty smiled and replied, "Thank you, Spot. I polish it every day."\n\nAfter playing with the car, Kitty and Spot felt thirsty. They found a small pond with clear water. They drank the water and felt very happy. They played together all day and became best friends.'

In [36]:
get_sample(dataset)

'Dave was a happy little boy. He went to the pier and saw a big, old ship. It was made of wood and was very big. Dave was so excited that he said, "Wow!"\n\nOn the dock, a elderly sailor was sitting in a chair next to the ship. He was wearing a big, white hat and had a long, grey beard. The sailor saw Dave and said, "Do you like my ship, little boy?"\n\nDave was surprised and he smiled and said, "Yes, I do. It\'s so big!"\n\nThe elderly sailor smiled and said, "Would you like to take a ride on it?"\n\nDave\'s face lit up. He said, "Yes, please!"\n\nThe sailor smiled and said, "Hop aboard, little one." Dave hopped onto the ship and the sailor snapped the sails open. They sailed way up into the sky and Dave saw all the clouds.\n\nHe had a wonderful day and when it was time to go home, the sailor snapped the sails closed and they sailed back to the pier. Dave waved goodbye to the elderly sailor and thanked him for the ride.'

In [38]:
tokenizer = Tokenizer(models.BPE(unk_token='[UNK]'))

tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=10000,
    special_tokens = ["[UNK]"],
    min_frequency = 2
)

tokenizer.train_from_iterator(dataset['validation']['text'], trainer)
tokenizer.get_vocab_size()

10000

In [43]:
a = tokenizer.encode("hi i wanna test this tokenizer to what to do for now")
a.tokens

['Ġhi',
 'Ġ',
 'i',
 'Ġwanna',
 'Ġtest',
 'Ġthis',
 'Ġto',
 'ken',
 'iz',
 'er',
 'Ġto',
 'Ġwhat',
 'Ġto',
 'Ġdo',
 'Ġfor',
 'Ġnow']

In [45]:
tokenizer.post_processor = processors.TemplateProcessing(
    single="<|endoftext|> $A <|endoftext|>",
    special_tokens=[("<|endoftext|>", tokenizer.token_to_id("<|endoftext|>"))],
)

TypeError: Expected Union[Tuple[str, int], Tuple[int, str], dict]

In [46]:
tokenizer.decoder = decoders.ByteLevel()

text = "Mr. Smith bought cheapsite.com for $1.5 million in 2023."
output = tokenizer.encode(text)

print("Tokens:", output.tokens)
print("Input IDs:", output.ids)

Tokens: ['ĠMr', '.', 'ĠSmith', 'Ġbought', 'Ġcheap', 'site', '.', 'com', 'Ġfor', 'Ġ', '$', '1', '.', '5', 'Ġmillion', 'Ġin', 'Ġ2', '0', '2', '3', '.']
Input IDs: [1605, 13, 4305, 1896, 3426, 9311, 13, 3461, 218, 97, 4, 16, 13, 20, 8024, 167, 7005, 15, 17, 18, 13]


In [47]:
tokenizer.get_vocab()

{'ĠAnd': 727,
 'ometer': 5492,
 'Ġreapp': 6690,
 'ÅĵNow': 8343,
 'Ġdresses': 5041,
 'ol': 1361,
 'Ġmassage': 5211,
 'lled': 8738,
 'ag': 414,
 'Ġadmiration': 7438,
 'Ġmoder': 3393,
 'Lilly': 6507,
 'Ġneighb': 1642,
 'come': 4029,
 'Ġshady': 7610,
 'Ġexcept': 7617,
 'aker': 5812,
 'Ġkeeping': 4737,
 'Ġmovie': 2665,
 'Ġafternoon': 3559,
 'Ġreads': 7282,
 'Ġlaptop': 9660,
 'ĠHarry': 4114,
 'Ġdisplay': 3803,
 'Ġknow': 553,
 'Sam': 1411,
 'Ġevery': 469,
 'Ġjudge': 4456,
 'Ġfrag': 2767,
 'Ġtaxi': 4087,
 'Ġpillow': 2315,
 'Ġaccident': 1660,
 'Ġsituation': 5838,
 'gu': 3302,
 'Ġdan': 794,
 'Ġkangaroos': 9995,
 'Ġsolution': 5508,
 'Ġdays': 1951,
 'Ġincredibly': 9160,
 'hen': 304,
 'Ġscampered': 7209,
 'Ġautomob': 5085,
 'Ġmon': 1058,
 'ĠHistory': 7911,
 'Ġpresented': 8370,
 'Ġsnowflakes': 8956,
 'Ġhockey': 4377,
 'Ġzigzag': 5107,
 'ĠBut': 406,
 'Ġknocks': 6648,
 'Ġsponge': 8068,
 'Ġener': 3595,
 'ĠNana': 8224,
 'Ġholidays': 9582,
 'U': 49,
 'Ġcake': 1022,
 'ĠNan': 6019,
 'Ġhall': 3997,
 'Ġsmile

In [48]:
tokenizer.decode(output.ids)

' Mr. Smith bought cheapsite.com for $1.5 million in 2023.'

In [49]:
output.ids

[1605,
 13,
 4305,
 1896,
 3426,
 9311,
 13,
 3461,
 218,
 97,
 4,
 16,
 13,
 20,
 8024,
 167,
 7005,
 15,
 17,
 18,
 13]

In [50]:
sample = get_sample(dataset)
sample

'One day, Grandpa and Grandma had a very special surprise for the little boy. They said "we have something very special for you! It is harmless tea!" The little boy had never had tea before, but Grandpa encouraged him to try some. He had a bit and found it to be very tasty. \n\nThe little boy smiled happily, enjoying the tea. He asked Grandpa if he could have some more. Grandpa smiled and gave him some more tea to enjoy. The little boy and Grandpa drank their tea together and Grandma smiled at them both.\n\nGrandpa hugged the little boy and encouraged him to enjoy the tea. The little boy smiled and thanked his Grandpa. Drinking the tea made him feel warm and happy on the inside. \n\nThe little boy enjoyed the harmless tea with his Grandpa and Grandma that day. They spent the rest of the afternoon talking, drinking tea and having a lot of fun.'

In [51]:
pprint(sample)
print(100*"-")

output = tokenizer.encode(sample)
print(output.tokens)
print(100*"-")

decoded_text = tokenizer.decode(output.ids)
pprint(decoded_text.strip())

('One day, Grandpa and Grandma had a very special surprise for the little boy. '
 'They said "we have something very special for you! It is harmless tea!" The '
 'little boy had never had tea before, but Grandpa encouraged him to try some. '
 'He had a bit and found it to be very tasty. \n'
 '\n'
 'The little boy smiled happily, enjoying the tea. He asked Grandpa if he '
 'could have some more. Grandpa smiled and gave him some more tea to enjoy. '
 'The little boy and Grandpa drank their tea together and Grandma smiled at '
 'them both.\n'
 '\n'
 'Grandpa hugged the little boy and encouraged him to enjoy the tea. The '
 'little boy smiled and thanked his Grandpa. Drinking the tea made him feel '
 'warm and happy on the inside. \n'
 '\n'
 'The little boy enjoyed the harmless tea with his Grandpa and Grandma that '
 'day. They spent the rest of the afternoon talking, drinking tea and having a '
 'lot of fun.')
------------------------------------------------------------------------------